In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder, MultiLabelBinarizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

#Подгрузка дф
df = pd.read_csv("credit_score_cleaned_train.csv")


#Удаление лишних колонок
cols_to_drop = ['id', 'customer_id', 'name', 'ssn', 'month']
df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True, errors='ignore')

#Обработка
def parse_loan_list(x):
    if pd.isna(x) or x == '[]' or x == "''" or x == '':
        return []
    try:
        x = x.strip("[]").replace("'", "").replace('"', '')
        return [item.strip() for item in x.split(',') if item.strip()]
    except:
        return []

if 'type_of_loan' in df.columns:
    df['type_of_loan_parsed'] = df['type_of_loan'].apply(parse_loan_list)
    mlb = MultiLabelBinarizer()
    loan_dummies = pd.DataFrame(mlb.fit_transform(df['type_of_loan_parsed']),
                                columns=[f'loan_{c}' for c in mlb.classes_],
                                index=df.index)
    df = pd.concat([df.drop(columns=['type_of_loan', 'type_of_loan_parsed'], errors='ignore'), loan_dummies], axis=1)
else:
    mlb = None

#Деление на нумерические и целевые колонки
numeric_cols = ['age', 'annual_income', 'monthly_inhand_salary', 'credit_history_age',
                'total_emi_per_month', 'num_bank_accounts', 'num_credit_card',
                'interest_rate', 'num_of_loan', 'delay_from_due_date',
                'num_of_delayed_payment', 'changed_credit_limit', 'num_credit_inquiries',
                'outstanding_debt', 'credit_utilization_ratio', 'amount_invested_monthly',
                'monthly_balance']
numeric_cols = [c for c in numeric_cols if c in df.columns]

for col in numeric_cols:
    df[col] = df[col].astype(str).str.replace(',', '').str.replace('%', '').str.strip()
    df[col] = pd.to_numeric(df[col], errors='coerce')

#Заполнение пропусков
for col in numeric_cols:
    med = df[col].median()
    df[col] = df[col].fillna(med)

categorical_cols = ['occupation', 'credit_mix', 'payment_of_min_amount', 'payment_behaviour']
categorical_cols = [c for c in categorical_cols if c in df.columns]
for col in categorical_cols:
    mode_val = df[col].mode()[0] if not df[col].mode().empty else 'Unknown'
    df[col] = df[col].fillna(mode_val)

#Таргет
target = 'credit_score'



#Удаление классов где 2 и меньше чтобы не было ситуаций где их не будет в x и в y
class_counts = df[target].value_counts()
rare_classes = class_counts[class_counts < 2].index.tolist()
if rare_classes:
    print(f"Удаляем редкие классы: {rare_classes} (всего {len(df[df[target].isin(rare_classes)])} строк)")
    df = df[~df[target].isin(rare_classes)]

# Кодируем целевую (если строка)
if df[target].dtype == 'object':
    le_target = LabelEncoder()
    df[target] = le_target.fit_transform(df[target])
else:
    le_target = None

print("\nРаспределение классов credit_score после очистки:")
print(df[target].value_counts())

#Очистка от выбросов
def remove_outliers_iqr(df, column, multiplier=1.5, verbose=True):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - multiplier * IQR
    upper = Q3 + multiplier * IQR
    initial_len = len(df)
    df_filtered = df[(df[column] >= lower) & (df[column] <= upper)]
    removed = initial_len - len(df_filtered)
    if verbose and removed > 0:
        print(f"  {column}: удалено {removed} строк ({removed/initial_len*100:.1f}%)")
    return df_filtered


df_clean = df.copy()
for col in numeric_cols:
    if col in df_clean.columns and col != target:
        df_clean = remove_outliers_iqr(df_clean, col, multiplier=1.5, verbose=True)

#Ванхот
cat_cols = ['occupation', 'credit_mix', 'payment_of_min_amount', 'payment_behaviour']
cat_cols = [c for c in cat_cols if c in df_clean.columns]

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
cat_encoded = encoder.fit_transform(df_clean[cat_cols])
cat_names = encoder.get_feature_names_out(cat_cols)
cat_df = pd.DataFrame(cat_encoded, columns=cat_names, index=df_clean.index)

df_clean = df_clean.drop(columns=cat_cols)
df_clean = pd.concat([df_clean, cat_df], axis=1)

#XY
y = df_clean[target]
X = df_clean.drop(columns=[target])

#Удаление Nan в дф
nan_y_count = y.isna().sum()
if nan_y_count > 0:
    mask = ~y.isna()
    y = y[mask]
    X = X[mask]


nan_X_rows = X.isna().any(axis=1).sum()
if nan_X_rows > 0:
    X = X.dropna()
    y = y[X.index]


#Масшатирование
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

#Выборка
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

#Обучение randomforest


model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Точность: {acc:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


#Сохранение артефактов
artifacts = {
    'model': model,
    'scaler': scaler,
    'onehot_encoder': encoder,
    'mlb': mlb,
    'feature_names': X.columns.tolist(),
    'label_encoder_target': le_target
}

with open('credit_model_complete.pkl', 'wb') as f:
    pickle.dump(artifacts, f)